In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import subprocess, os
#from Utils.utils import visualize_hcube
import h5py
import matplotlib.pyplot as plt
from matplotlib.transforms import Affine2D
import h5py, re, tqdm
from collections import Counter
import tqdm

In [2]:
# Mounting dataset from D: to WSL 
WINDOWS_DRIVE = "E:"   
WINDOWS_PATH = r"E:\HSI_Dataset_2\Elements\data"
LINK_NAME     = "data_external"  

# Mount drive if missing
mnt_path = Path(f"/mnt/{WINDOWS_DRIVE[0].lower()}")
#if not mnt_path.exists():
print(f"[INFO] Mounting {WINDOWS_DRIVE} into {mnt_path} ...")
subprocess.run(["sudo", "mkdir", "-p", str(mnt_path)], check=True)
res = subprocess.run(["sudo", "mount", "-t", "drvfs", WINDOWS_DRIVE, str(mnt_path)], capture_output=True, text=True)
if res.returncode != 0:
    raise RuntimeError(f"Failed to mount {WINDOWS_DRIVE}: {res.stderr}")
#else:
#    print(f"[OK] {mnt_path} already exists")

# Verify dataset path 
dataset_path = Path(str(WINDOWS_PATH).replace("\\", "/").replace(":", "").replace("E", "/mnt/e", 1))
if not dataset_path.exists():
    print(f"[WARN] Dataset not found at {dataset_path}. Let's check what's under /mnt/e:")
    os.system("ls -la /mnt/d")
    raise FileNotFoundError("Fix dataset path above and rerun.")
print(f"[OK] Found dataset: {dataset_path}")

# Create symlink inside project
proj_root = Path.cwd()
link_path = proj_root / LINK_NAME
if link_path.exists() or link_path.is_symlink():
    print(f"[INFO] Removing old link {link_path}")
    link_path.unlink()
link_path.symlink_to(dataset_path, target_is_directory=True)
print(f"[OK] Linked {link_path} -> {dataset_path}")

# Show a few sample files for confirmation
import itertools
exts = {".hdf5", ".h5", ".hdr", ".tif", ".tiff"}
found = list(itertools.islice((p for p in link_path.rglob("*") if p.suffix.lower() in exts), 10))
if found:
    print("Sample files:")
    for f in found: print("  ", f.relative_to(link_path))
else:
    print("No .hdf5/.h5/.hdr/.tif files found yet — check deeper folders.")

[INFO] Mounting E: into /mnt/e ...


[OK] Found dataset: /mnt/e/HSI_Dataset_2/Elements/data
[INFO] Removing old link /home/isaacmuscat/Grain-Variety-Classification/data_external
[OK] Linked /home/isaacmuscat/Grain-Variety-Classification/data_external -> /mnt/e/HSI_Dataset_2/Elements/data
Sample files:
   raw/calibration/FX10/calibration_1705a_0.hdf5
   raw/calibration/FX10/calibration_1705a_1.hdf5
   raw/calibration/FX10/calibration_1705a_2.hdf5
   raw/calibration/FX10/calibration_1705a_3.hdf5
   raw/calibration/FX10/calibration_1705_0.hdf5
   raw/calibration/FX10/calibration_1705_1.hdf5
   raw/calibration/FX10/calibration_1705_2.hdf5
   raw/calibration/FX10/calibration_1705_3.hdf5
   raw/calibration/Snapshot/processed/calibration_1705a_0/image_0000000003.hdr
   raw/calibration/Snapshot/processed/calibration_1705a_1/image_0000000000.hdr


In [3]:
ROOT = Path("data_external")  
HR_ROOT = ROOT / "raw" / "FX10" 

OUTDIR = HR_ROOT.parent.parent / "processed" / "quickrun"  
OUTDIR.mkdir(parents=True, exist_ok=True)

print("ROOT:    ", ROOT.resolve())
print("Basepath:", HR_ROOT.resolve())
print("OUTDIR:  ", OUTDIR.resolve())


ROOT:     /mnt/e/HSI_Dataset_2/Elements/data
Basepath: /mnt/e/HSI_Dataset_2/Elements/data/raw/FX10
OUTDIR:   /mnt/e/HSI_Dataset_2/Elements/data/processed/quickrun


In [4]:
# Look for all files in the folder that end with .hdf5 
df = pd.DataFrame({"filepath_FX10": list(Path(f"{HR_ROOT}").rglob("**/*.hdf5"))})
# Give the sample name 
df['sample_name'] = df.filepath_FX10.apply(lambda x : x.stem)

df

,filepath_FX10,sample_name
0,data_external/raw/FX10/barley_l0.hdf5,barley_l0
1,data_external/raw/FX10/barley_l1.hdf5,barley_l1
2,data_external/raw/FX10/barley_l2.hdf5,barley_l2
3,data_external/raw/FX10/barley_l3.hdf5,barley_l3
4,data_external/raw/FX10/barley_l4.hdf5,barley_l4
...,...,...
175,data_external/raw/FX10/wheatgrass_s0.hdf5,wheatgrass_s0
176,data_external/raw/FX10/wheatgrass_s1.hdf5,wheatgrass_s1
177,data_external/raw/FX10/wheatgrass_s2.hdf5,wheatgrass_s2
178,data_external/raw/FX10/wheatgrass_s3.hdf5,wheatgrass_s3


In [ ]:
# --- Verifying amount of valid .hdf5 files found ---
all_files = list(HR_ROOT.rglob("*.hdf5"))
print("Total .hdf5 files found:", len(all_files))

df_files = pd.DataFrame({"filepath_FX10": all_files})
df_files["sample_name"] = df_files["filepath_FX10"].apply(lambda p: p.stem)


def resolve_valid_hdf5(path: Path) -> Path | None:
    """
    Returns:
        - Path to a valid file or None if it fails
    """
    path = Path(path)

    try:
        with h5py.File(path, "r"):
            return path
    except Exception:
        return None

df_files["resolved_path"] = df_files["filepath_FX10"].apply(resolve_valid_hdf5)

df_valid = df_files[df_files["resolved_path"].notna()].copy()
df_invalid = df_files[df_files["resolved_path"].isna()].copy()

print("Valid HDF5 files:", len(df_valid))
print("Invalid/unreadable files:", len(df_invalid))

if not df_invalid.empty:
    print("\nUnreadable sample_names:")
    print(df_invalid["sample_name"].tolist())


Total .hdf5 files found: 180
Valid HDF5 files: 180
Invalid/unreadable files: 0


In [6]:
def load_cube(path, verbose=False):

    with h5py.File(path, 'r') as f:
        f = h5py.File(path)

        if verbose:
            print("What is inside an h5 file?\n")
            for key in f.keys(): print(f"key:'{key}', \nin which there is '{f[key]}' \
            \nwhich we load in a numpy array of shape: {np.array(f[key]).shape}\n");
        
            print(f"Selected to load cube from file:\n{path}")

        hcube = np.array(f['hypercube'][:,:,:])/10000  #.astype('float32')
        darkref = np.array(f['dark_reference'])/10000 #.astype('float32')
        whiteref = np.array(f['white_reference'])/10000
       
    
        hcube = np.swapaxes(hcube,-1,0).astype('float32')
        hcube = np.fliplr(hcube)
       
        wlens = f['hypercube'].attrs['wavelength_nm']

    return hcube, wlens, darkref, whiteref

In [7]:
# --- Defining files to be used for training and testing, based on train-test split of synthetic image generation --- 

original_test = ["barley_l4", "barley_s4", "buckwheat_m4", "corn_l4", "corn_s4", "flax_m4", "flaxb_l4", "flaxb_s4", "millet_m4",
                 "mix_l4", "mix_s4", "pumpkin_m4", "rye_l4", "rye_s4", "spelt_m4", "sunflower_l4", "sunflower_s4", "wheatgrass_m4"]

mask_test = df_valid["sample_name"].isin(original_test)

test_files = df_valid[mask_test]
train_files = df_valid[~mask_test]

train_files

,filepath_FX10,sample_name,resolved_path
0,data_external/raw/FX10/barley_l0.hdf5,barley_l0,data_external/raw/FX10/barley_l0.hdf5
1,data_external/raw/FX10/barley_l1.hdf5,barley_l1,data_external/raw/FX10/barley_l1.hdf5
2,data_external/raw/FX10/barley_l2.hdf5,barley_l2,data_external/raw/FX10/barley_l2.hdf5
3,data_external/raw/FX10/barley_l3.hdf5,barley_l3,data_external/raw/FX10/barley_l3.hdf5
5,data_external/raw/FX10/barley_m0.hdf5,barley_m0,data_external/raw/FX10/barley_m0.hdf5
...,...,...,...
175,data_external/raw/FX10/wheatgrass_s0.hdf5,wheatgrass_s0,data_external/raw/FX10/wheatgrass_s0.hdf5
176,data_external/raw/FX10/wheatgrass_s1.hdf5,wheatgrass_s1,data_external/raw/FX10/wheatgrass_s1.hdf5
177,data_external/raw/FX10/wheatgrass_s2.hdf5,wheatgrass_s2,data_external/raw/FX10/wheatgrass_s2.hdf5
178,data_external/raw/FX10/wheatgrass_s3.hdf5,wheatgrass_s3,data_external/raw/FX10/wheatgrass_s3.hdf5


In [8]:
# --- Defining a few new columns for the HR train files ---

train_files.loc[:, 'variety'] = train_files['sample_name'].str.split("_").str[0]
train_files.loc[:, 'batch']   = train_files['sample_name'].str.split("_").str[-1]

train_files.loc[:, 'size'] = train_files['batch'].str[0]          
train_files.loc[:, 'rep']  = train_files['batch'].str[1:].astype(int)   

/tmp/ipykernel_23331/2830062965.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_files.loc[:, 'variety'] = train_files['sample_name'].str.split("_").str[0]
/tmp/ipykernel_23331/2830062965.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_files.loc[:, 'batch']   = train_files['sample_name'].str.split("_").str[-1]
/tmp/ipykernel_23331/2830062965.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value ins

In [9]:
# --- Load LR HSI Data, and defining a few new columns for the LR train files ---
LR_ROOT = ROOT / "raw" / "Snapshot" / "unprocessed"

rows = []

# Regex to match only valid case folders: e.g. barley_m_unprocessed
case_regex = re.compile(r"^([a-zA-Z]+)_(s|m|l)_unprocessed$")

for case_dir in sorted(LR_ROOT.iterdir()):
    if not case_dir.is_dir():
        continue

    m = case_regex.match(case_dir.name)
    if m is None:
        # Skip folders like barley_mx_unprocessed, barley_l_old, etc.
        continue

    variety, size = m.group(1), m.group(2)

    # Five repetition folders inside each case
    for subdir in sorted(case_dir.iterdir()):
        if not subdir.is_dir() or subdir.name == "context":
            continue

        # Expect subfolder format: barley_m0, barley_m1, ..., barley_m4
        # Subfolder must start with "<variety>_<size>"
        if not subdir.name.startswith(f"{variety}_{size}"):
            continue

        # Extract repetition number from the very end of the folder
        if not subdir.name[-1].isdigit():
            continue
        rep = int(subdir.name[-1])

        # Find RAW file inside the repetition folder
        raw_candidates = list(subdir.glob("*.RAW")) + list(subdir.glob("*.raw"))
        if len(raw_candidates) == 0:
            continue

        raw_path = raw_candidates[0]

        # Build a sample_name matching HR convention
        # e.g. barley_m4
        sample_name = f"{variety}_{size}{rep}"

        rows.append({
            "filepath_snapshot": raw_path,
            "variety": variety,
            "size": size,
            "rep": rep,
            "sample_folder": subdir.name,
            "sample_name": sample_name,
            "case_name": case_dir.name,
        })

snapshot_df = pd.DataFrame(rows)
print("Number of filtered LR snapshots:", len(snapshot_df))
snapshot_df



Number of filtered LR snapshots: 180


,filepath_snapshot,variety,size,rep,sample_folder,sample_name,case_name
0,data_external/raw/Snapshot/unprocessed/barley_...,barley,l,0,barley_l0,barley_l0,barley_l_unprocessed
1,data_external/raw/Snapshot/unprocessed/barley_...,barley,l,1,barley_l1,barley_l1,barley_l_unprocessed
2,data_external/raw/Snapshot/unprocessed/barley_...,barley,l,2,barley_l2,barley_l2,barley_l_unprocessed
3,data_external/raw/Snapshot/unprocessed/barley_...,barley,l,3,barley_l3,barley_l3,barley_l_unprocessed
4,data_external/raw/Snapshot/unprocessed/barley_...,barley,l,4,barley_l4,barley_l4,barley_l_unprocessed
...,...,...,...,...,...,...,...
175,data_external/raw/Snapshot/unprocessed/wheatgr...,wheatgrass,s,0,wheatgrass_s0,wheatgrass_s0,wheatgrass_s_unprocessed
176,data_external/raw/Snapshot/unprocessed/wheatgr...,wheatgrass,s,1,wheatgrass_s1,wheatgrass_s1,wheatgrass_s_unprocessed
177,data_external/raw/Snapshot/unprocessed/wheatgr...,wheatgrass,s,2,wheatgrass_s2,wheatgrass_s2,wheatgrass_s_unprocessed
178,data_external/raw/Snapshot/unprocessed/wheatgr...,wheatgrass,s,3,wheatgrass_s3,wheatgrass_s3,wheatgrass_s_unprocessed


In [10]:
# --- Similarly to the HR data before, defining files to be used for training and testing ---
mask_test_lr = snapshot_df["sample_name"].isin(original_test)

snapshot_test_files = snapshot_df[mask_test_lr].copy()
snapshot_train_files = snapshot_df[~mask_test_lr].copy()

snapshot_train_files

,filepath_snapshot,variety,size,rep,sample_folder,sample_name,case_name
0,data_external/raw/Snapshot/unprocessed/barley_...,barley,l,0,barley_l0,barley_l0,barley_l_unprocessed
1,data_external/raw/Snapshot/unprocessed/barley_...,barley,l,1,barley_l1,barley_l1,barley_l_unprocessed
2,data_external/raw/Snapshot/unprocessed/barley_...,barley,l,2,barley_l2,barley_l2,barley_l_unprocessed
3,data_external/raw/Snapshot/unprocessed/barley_...,barley,l,3,barley_l3,barley_l3,barley_l_unprocessed
5,data_external/raw/Snapshot/unprocessed/barley_...,barley,m,0,barley_m0,barley_m0,barley_m_unprocessed
...,...,...,...,...,...,...,...
175,data_external/raw/Snapshot/unprocessed/wheatgr...,wheatgrass,s,0,wheatgrass_s0,wheatgrass_s0,wheatgrass_s_unprocessed
176,data_external/raw/Snapshot/unprocessed/wheatgr...,wheatgrass,s,1,wheatgrass_s1,wheatgrass_s1,wheatgrass_s_unprocessed
177,data_external/raw/Snapshot/unprocessed/wheatgr...,wheatgrass,s,2,wheatgrass_s2,wheatgrass_s2,wheatgrass_s_unprocessed
178,data_external/raw/Snapshot/unprocessed/wheatgr...,wheatgrass,s,3,wheatgrass_s3,wheatgrass_s3,wheatgrass_s_unprocessed
